# Reprodução do House Prices da tese

Recorte do `referencia_lusquino/regression/houseprice/houseprice.py`, mantendo
só o caminho que leva ao resultado da Regression WiSARD. O objetivo não é bater
o número, é **confirmar que entendemos o pipeline dele**.

## O que foi mantido, e é o que importa

O pré-processamento inteiro, linha por linha: as três políticas de imputação, a
correção de assimetria por `boxcox1p`, os atributos derivados, o one-hot, a
remoção de outliers, o `overfit_reducer` e o split `test_size=0.33,
random_state=0`.

E, principalmente, o **`np.log1p` no alvo**, que é a transformação que explica o
0.278 da Tabela 5.3 da tese.

## O que foi tirado, e por quê

| retirado | motivo |
|---|---|
| gráficos exploratórios (linhas 60-100) | é a parte que usa `seaborn` com API removida nas versões novas; não afeta resultado |
| XGBoost, Random Forest, Decision Tree | não estão instalados e não são o objeto agora |
| ClusRegressionWiSARD | é o que está ativo no fim do arquivo original, mas não é o modelo em questão |
| varredura completa | `range(5,31)` × todos os divisores × 7 médias × 10 repetições dá umas 70 mil execuções, algo entre 4 e 20 horas |

## Sobre os dados

Usa os dois arquivos originais do Kaggle, `train.csv` (1460 linhas, com rótulo) e
`test.csv` (1459 linhas, sem rótulo), como o script original. Eles vêm de um
repositório público que os distribui, e foram conferidos contra o que o OpenML
entrega: mesmas colunas, mesmos `Id`, mesmo `SalePrice`.

Como no original, os dois são concatenados em `all_data` **antes** da imputação.
Isso significa que as estatísticas de imputação, as colunas do one-hot e a
correção de assimetria enxergam o conjunto de teste. Isso é vazamento, e está
assim porque o objetivo aqui é reproduzir o pipeline dele, não corrigi-lo.

O `test.csv` nunca entra na avaliação: o `X_sub`, que seria a parte dele, é
definido no original e nunca usado em experimento nenhum.

In [16]:
import sys
!{sys.executable} -m pip install pybind11
!{sys.executable} -m pip install --no-build-isolation "git+https://github.com/IAZero/wisardpkg.git@develop"

  Cloning https://github.com/IAZero/wisardpkg.git (to revision develop) to /private/var/folders/kn/t8jqs6cn3gs41cylbls499h80000gn/T/pip-req-build-1t1hhw4x
  Running command git clone --filter=blob:none --quiet https://github.com/IAZero/wisardpkg.git /private/var/folders/kn/t8jqs6cn3gs41cylbls499h80000gn/T/pip-req-build-1t1hhw4x
  Running command git checkout -b develop --track origin/develop
  Switched to a new branch 'develop'
  branch 'develop' set up to track 'origin/develop'.
  Resolved https://github.com/IAZero/wisardpkg.git to commit c51b395c0e85ae007bb7f8699515a3136356ef65
  Preparing metadata (pyproject.toml) ... done


In [17]:
import numpy as np
import pandas as pd
import time

from scipy.stats import skew, boxcox_normmax
from scipy.special import boxcox1p
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

import wisardpkg as wsd
print("wisardpkg", wsd.__version__)

wisardpkg 2.0.0a7


---
## 1. Carregamento e transformação do alvo

Duas linhas do original que importam muito:

```python
train = train[train.GrLivArea < 4500]              # remove 2 casas
train["SalePrice"] = np.log1p(train["SalePrice"])  # ALVO EM ESCALA LOG
```

O `log1p` é `log(1 + x)`. É por isso que o MAE reportado na tese é 0.278 e não
um valor em dólares: ele está em espaço logarítmico.

In [18]:
BASE = "https://raw.githubusercontent.com/ankita1112/House-Prices-Advanced-Regression/master/"

train = pd.read_csv(BASE + "train.csv")
test  = pd.read_csv(BASE + "test.csv")
print(f"train.csv: {train.shape}   test.csv: {test.shape}  (test sem SalePrice)")

# linha 73 do original: só o TREINO é filtrado
train = train[train.GrLivArea < 4500]
train.reset_index(drop=True, inplace=True)
print(f"após GrLivArea < 4500: {train.shape}")

# linha 82 do original: A TRANSFORMAÇÃO DO ALVO
train["SalePrice"] = np.log1p(train["SalePrice"])
print(f"\nalvo depois do log1p: {train.SalePrice.min():.4f} a {train.SalePrice.max():.4f}")
print(f"  (em dólares isso é ${np.expm1(train.SalePrice.min()):,.0f} "
      f"a ${np.expm1(train.SalePrice.max()):,.0f})")

train.drop(columns=["Id"], axis=1, inplace=True)
test.drop(columns=["Id"], axis=1, inplace=True)

y = train["SalePrice"].reset_index(drop=True)

# linha 106 do original: os dois juntos ANTES de imputar
all_data = pd.concat((train, test)).reset_index(drop=True)
all_data.drop(["SalePrice"], axis=1, inplace=True)
print(f"\nall_data (treino + teste): {all_data.shape}")
print(f"  as primeiras {len(y)} linhas são o treino rotulado; o resto é o test.csv")

train.csv: (1460, 81)   test.csv: (1459, 80)  (test sem SalePrice)
após GrLivArea < 4500: (1458, 81)

alvo depois do log1p: 10.4603 a 13.5345
  (em dólares isso é $34,900 a $755,000)

all_data (treino + teste): (2917, 79)
  as primeiras 1458 linhas são o treino rotulado; o resto é o test.csv


---
## 2. Imputação

Três políticas diferentes, e a terceira é a que a tese não documenta:

**Categóricas onde ausência tem significado** recebem a string `'None'`. Sem
garagem, sem porão, sem lareira: o NaN não é dado faltante, é a informação de que
aquilo não existe.

**Numéricas de área e porão** recebem `0`, pela mesma lógica.

**`LotFrontage`** recebe a **média do bairro**, não a mediana global. Casas do
mesmo `Neighborhood` têm testada parecida, então a média local é uma estimativa
melhor.

In [19]:
missing_val_col = ["Alley", "PoolQC", "MiscFeature", "Fence", "FireplaceQu",
                   "GarageType", "GarageFinish", "GarageQual", "GarageCond",
                   "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
                   "BsmtFinType2", "MasVnrType"]
for i in missing_val_col:
    all_data[i] = all_data[i].fillna("None")

missing_val_col2 = ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
                    "BsmtFullBath", "BsmtHalfBath", "GarageYrBlt", "GarageArea",
                    "GarageCars", "MasVnrArea"]
for i in missing_val_col2:
    all_data[i] = all_data[i].fillna(0)

# média do BAIRRO, não mediana global
all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.mean()))

# viram texto de propósito: são códigos, não quantidades
all_data["MSSubClass"] = all_data["MSSubClass"].astype(str)
all_data["MSZoning"] = all_data.groupby("MSSubClass")["MSZoning"].transform(
    lambda x: x.fillna(x.mode()[0]))
all_data["YrSold"] = all_data["YrSold"].astype(str)
all_data["MoSold"] = all_data["MoSold"].astype(str)

all_data["Functional"]  = all_data["Functional"].fillna("Typ")
all_data["Utilities"]   = all_data["Utilities"].fillna("AllPub")
all_data["Exterior1st"] = all_data["Exterior1st"].fillna(all_data["Exterior1st"].mode()[0])
all_data["Exterior2nd"] = all_data["Exterior2nd"].fillna(all_data["Exterior2nd"].mode()[0])
all_data["KitchenQual"] = all_data["KitchenQual"].fillna("TA")
all_data["SaleType"]    = all_data["SaleType"].fillna(all_data["SaleType"].mode()[0])
all_data["Electrical"]  = all_data["Electrical"].fillna("SBrkr")

falta = all_data.isna().sum()
print(f"colunas com NaN restante: {(falta > 0).sum()}")
if (falta > 0).any():
    print(falta[falta > 0].to_string())

colunas com NaN restante: 0


---
## 3. Correção de assimetria e atributos derivados

Colunas numéricas com assimetria acima de 0,5 passam por `boxcox1p`, uma
transformação que aproxima a distribuição de uma normal. Depois entram nove
atributos derivados, somando áreas e criando indicadores binários.

Repare que `MSSubClass`, `YrSold` e `MoSold` já viraram texto na célula anterior,
então não entram aqui e vão para o one-hot.

In [20]:
def fixing_skewness(df):
    """Igual ao original, com um try/except a mais.

    O `boxcox_normmax` usa o método `pearsonr` por padrão, e em versões novas do
    scipy o otimizador dele falha com BracketError em algumas colunas. Aqui a
    coluna que falha fica SEM transformar, o que é mais seguro que o método
    alternativo `mle`: para `YearRemodAdd` ele devolve lambda 40.6, que produz
    valores astronômicos.
    """
    puladas = []
    numeric_feats = df.dtypes[df.dtypes != "object"].index
    skewed = df[numeric_feats].apply(lambda x: skew(x)).sort_values(ascending=False)
    alvo = skewed[abs(skewed) > 0.5].index
    for feat in alvo:
        try:
            df[feat] = boxcox1p(df[feat], boxcox_normmax(df[feat] + 1))
        except Exception:
            puladas.append(feat)
    return len(alvo), puladas


n_alvo, puladas = fixing_skewness(all_data)
print(f"boxcox aplicado em {n_alvo - len(puladas)} de {n_alvo} colunas assimétricas")
if puladas:
    print(f"  sem transformar (BracketError do scipy): {puladas}")

all_data = all_data.drop(["Utilities", "Street", "PoolQC"], axis=1)

all_data["TotalSF"]       = all_data["TotalBsmtSF"] + all_data["1stFlrSF"] + all_data["2ndFlrSF"]
all_data["YrBltAndRemod"] = all_data["YearBuilt"] + all_data["YearRemodAdd"]
all_data["Total_sqr_footage"] = (all_data["BsmtFinSF1"] + all_data["BsmtFinSF2"] +
                                 all_data["1stFlrSF"] + all_data["2ndFlrSF"])
all_data["Total_Bathrooms"]   = (all_data["FullBath"] + 0.5 * all_data["HalfBath"] +
                                 all_data["BsmtFullBath"] + 0.5 * all_data["BsmtHalfBath"])
all_data["Total_porch_sf"]    = (all_data["OpenPorchSF"] + all_data["3SsnPorch"] +
                                 all_data["EnclosedPorch"] + all_data["ScreenPorch"] +
                                 all_data["WoodDeckSF"])

all_data["haspool"]       = (all_data["PoolArea"]    > 0).astype(int)
all_data["has2ndfloor"]   = (all_data["2ndFlrSF"]    > 0).astype(int)
all_data["hasgarage"]     = (all_data["GarageArea"]  > 0).astype(int)
all_data["hasbsmt"]       = (all_data["TotalBsmtSF"] > 0).astype(int)
all_data["hasfireplace"]  = (all_data["Fireplaces"]  > 0).astype(int)

final_features = pd.get_dummies(all_data).reset_index(drop=True)
print(f"depois do one-hot: {final_features.shape}")

/Users/helenaburock/Git/estudos_ml/.venv/lib/python3.9/site-packages/scipy/stats/_morestats.py:1298: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, prob = _stats_py.pearsonr(xvals, yvals)
/Users/helenaburock/Git/estudos_ml/.venv/lib/python3.9/site-packages/scipy/stats/_morestats.py:1298: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, prob = _stats_py.pearsonr(xvals, yvals)


boxcox aplicado em 25 de 27 colunas assimétricas
  sem transformar (BracketError do scipy): ['LotArea', '1stFlrSF']
depois do one-hot: (2917, 333)


---
## 4. Outliers, `overfit_reducer` e split

O `overfit_reducer` descarta colunas em que um único valor aparece em mais de
**99,94%** das linhas. É um corte muito mais rigoroso que o nosso, que usava 96%
de zeros.

O split é `test_size=0.33, random_state=0`. Com 1453 linhas isso dá 973 de treino
e 480 de teste, que é exatamente o que a tese reporta.

In [21]:
X = final_features.iloc[:len(y), :]

outliers = [30, 88, 462, 631, 1322]
X = X.drop(X.index[outliers])
y_ = y.drop(y.index[outliers])

def overfit_reducer(df):
    return [c for c in df.columns
            if df[c].value_counts().iloc[0] / len(df) * 100 > 99.94]

descartadas = overfit_reducer(X)
X = X.drop(descartadas, axis=1)
print(f"overfit_reducer descartou {len(descartadas)} colunas -> {X.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y_, test_size=.33, random_state=0)
X_train, X_test = X_train.values.astype(float), X_test.values.astype(float)
y_train, y_test = y_train.values, y_test.values

print(f"\ntreino {len(X_train)}   teste {len(X_test)}")
print(f"a tese reporta:  treino 973   teste 480")
print(f"\natributos: {X_train.shape[1]}")
print(f"alvo (log1p): treino {y_train.min():.3f} a {y_train.max():.3f}")

overfit_reducer descartou 1 colunas -> (1453, 332)

treino 973   teste 480
a tese reporta:  treino 973   teste 480

atributos: 332
alvo (log1p): treino 10.460 a 13.534


---
## 5. Referências de erro

Antes de rodar a ReW, vale saber o que é um erro bom e o que é um erro ruim
**nesta escala**. Tudo aqui está em `log1p`, não em dólares.

In [22]:
mae_media   = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))
mae_mediana = mean_absolute_error(y_test, np.full(len(y_test), np.median(y_train)))

print("REFERÊNCIAS (em escala log1p)\n")
print(f"  chutar a média do treino   : MAE {mae_media:.4f}")
print(f"  chutar a mediana do treino : MAE {mae_mediana:.4f}   <- é o que precisa ser batido")
print(f"\n  tese, Tabela 5.3, ReW      : MAE 0.278")
print(f"  tese, Tabela 5.3, CReW     : MAE 0.194")
print(f"  tese, Tabela 5.3, nTRN     : MAE 0.302")
print(f"\nEm erro relativo, 0.278 em log equivale a um fator de "
      f"{np.expm1(0.278):.1%} de erro típico.")

REFERÊNCIAS (em escala log1p)

  chutar a média do treino   : MAE 0.3151
  chutar a mediana do treino : MAE 0.3156   <- é o que precisa ser batido

  tese, Tabela 5.3, ReW      : MAE 0.278
  tese, Tabela 5.3, CReW     : MAE 0.194
  tese, Tabela 5.3, nTRN     : MAE 0.302

Em erro relativo, 0.278 em log equivale a um fator de 32.0% de erro típico.


---
## 6. A varredura completa, nas duas escalas do alvo

### O que a literatura publica, e o que não publica

O artigo do Neurocomputing 416 (2020), Tabela 2, e a Tabela 5.3 da tese trazem o
mesmo resultado:

| modelo | House Prices | média vencedora |
|---|---|---|
| ReW | 0.278 ± 0 | **Median** |
| CReW | 0.194 ± 0 | Power Mean |
| nTRN | 0.302 ± 0 | — |

O **tipo de média vencedor** está publicado. O **thermometer size** e o
**address size** vencedores não estão, em nenhum dos dois. O que existe é a
descrição do espaço de busca, idêntica nos dois textos:

> "The thermometer had its size varied from 5 to 30 bits and the tuple address
> size range was calculated according to the size of the thermometer (all values
> divisible by data word length from 2 bits were used)."

Isso corresponde ao `allDividers(sTherm)` do `houseprice.py`, e não ao
`range(sqrt(sTherm), sTherm/2 + 1)` do `old_houseprice.py`. É o que fixa qual dos
dois scripts reproduzir.

Nenhum dos dois textos informa a **escala do alvo**. Por isso a varredura roda
duas vezes.

### As duas escalas

| escala | de onde vem | conversão possível? |
|---|---|---|
| `log1p` | linha 82 do `houseprice.py`: `np.log1p(SalePrice)` | não, log não é linear |
| `÷ 1e5` | informado pelo autor | sim para quase todas as médias, **não** para a `ExponentialMean` |

A `ExponentialMean` tem `exp` na fórmula e não é equivariante por escala, então
não dá para converter o MAE no fim. As duas varreduras treinam de verdade, cada
uma com seu alvo.

### Custo

426 pares (thermometer, address) × 7 médias × 10 repetições, duas vezes.
Aproximadamente 100 minutos no total. Grava resultado parcial em CSV a cada
configuração, então dá para acompanhar sem esperar o fim.

In [23]:
import os, time

def allDividers(value):
    div = [i for i in range(2, int(value / 2) + 1) if value % i == 0]
    div.append(value)
    return div


def para_lista(bin_input):
    """BinInput -> lista de inteiros. Nesta versão do wisardpkg o DataSet não
    aceita BinInput direto: dá RuntimeError "Index out of range!"."""
    return [bin_input[i] for i in range(len(bin_input))]


# as sete médias, na mesma ordem do script original
def medias():
    return {"PowerMean(2)": wsd.PowerMean(2), "HarmonicMean": wsd.HarmonicMean(),
            "HarmonicPower(2)": wsd.HarmonicPowerMean(2), "GeometricMean": wsd.GeometricMean(),
            "ExponentialMean": wsd.ExponentialMean(), "SimpleMean": wsd.SimpleMean(),
            "Median": wsd.Median()}

THERMS     = range(5, 31)   # igual ao original
REPETICOES = 10             # igual ao original
LIMITE_ADDR = 64            # acima disso o índice estoura o inteiro de 64 bits
                            # do getIndex e o resultado vira lixo (MAE > 8)

def varredura(y_tr, y_te, rotulo, arquivo):
    """Varredura completa do houseprice.py, para uma escala de alvo."""
    mins = [np.min(X_train[:, i]) for i in range(X_train.shape[1])]
    maxs = [np.max(X_train[:, i]) for i in range(X_train.shape[1])]
    linhas, t0 = [], time.perf_counter()

    with open(arquivo, "w") as f:
        f.write("escala,therm,address,media,n_rams,MAE,MAE_dp\n")

    for t in THERMS:
        therm = [t] * X_train.shape[1]
        sTherm = int(np.sum(therm))
        DT = wsd.DynamicThermometer(therm, mins, maxs)
        DStr = wsd.DataSet([para_lista(DT.transform(X_train[i])) for i in range(len(X_train))], y_tr)
        DSte = wsd.DataSet([para_lista(DT.transform(X_test[i]))  for i in range(len(X_test))],  y_te)

        for addr in [d for d in allDividers(sTherm) if d <= LIMITE_ADDR]:
            for nome, m in medias().items():
                erros = []
                for _ in range(REPETICOES):
                    rw = wsd.RegressionWisard(addressSize=addr, mean=m)
                    rw.train(DStr)
                    out = np.array(rw.predict(DSte))
                    out[np.isnan(out)] = np.nanmean(out)   # como no original
                    out[out == np.inf] = 0
                    erros.append(mean_absolute_error(out, y_te))
                reg = {"escala": rotulo, "therm": t, "address": addr, "media": nome,
                       "n_rams": sTherm // addr,
                       "MAE": float(np.mean(erros)), "MAE_dp": float(np.std(erros))}
                linhas.append(reg)
                with open(arquivo, "a") as f:
                    f.write(f"{rotulo},{t},{addr},{nome},{reg['n_rams']},"
                            f"{reg['MAE']:.6f},{reg['MAE_dp']:.6f}\n")
        print(f"  [{rotulo}] therm={t:>2} concluído  "
              f"({len(linhas)} configs, {(time.perf_counter()-t0)/60:.1f} min)")

    return pd.DataFrame(linhas)

print(f"espaço de busca: {sum(len([d for d in allDividers(t*X_train.shape[1]) if d <= LIMITE_ADDR]) for t in THERMS)} "
      f"pares (therm, address) x 7 médias x {REPETICOES} repetições")

espaço de busca: 185 pares (therm, address) x 7 médias x 10 repetições


### Escala 1: `log1p`, como está no código publicado

O alvo já veio transformado da célula de carregamento, então `y_train` e `y_test`
estão em `log1p`.

In [24]:
res_log = varredura(y_train, y_test, "log1p", "varredura_log1p.csv")
print(f"\nconcluído: {len(res_log)} configurações")

  [log1p] therm= 5 concluído  (35 configs, 0.4 min)
  [log1p] therm= 6 concluído  (84 configs, 1.2 min)
  [log1p] therm= 7 concluído  (119 configs, 1.7 min)
  [log1p] therm= 8 concluído  (154 configs, 2.4 min)
  [log1p] therm= 9 concluído  (210 configs, 3.5 min)
  [log1p] therm=10 concluído  (259 configs, 4.5 min)
  [log1p] therm=11 concluído  (294 configs, 5.4 min)
  [log1p] therm=12 concluído  (357 configs, 7.1 min)
  [log1p] therm=13 concluído  (392 configs, 8.2 min)
  [log1p] therm=14 concluído  (441 configs, 9.8 min)
  [log1p] therm=15 concluído  (518 configs, 14.0 min)
  [log1p] therm=16 concluído  (560 configs, 15.5 min)
  [log1p] therm=17 concluído  (588 configs, 16.8 min)
  [log1p] therm=18 concluído  (658 configs, 20.0 min)
  [log1p] therm=19 concluído  (686 configs, 21.6 min)
  [log1p] therm=20 concluído  (742 configs, 24.5 min)
  [log1p] therm=21 concluído  (812 configs, 28.4 min)
  [log1p] therm=22 concluído  (854 configs, 31.1 min)
  [log1p] therm=23 concluído  (882 confi

### Escala 2: preço dividido por $10^5$

Desfaz o `log1p` com `expm1`, voltando aos dólares, e divide por $10^5$. Treina
de novo, porque a `ExponentialMean` não permite converter o MAE no fim.

In [25]:
y_train_d = np.expm1(y_train) / 1e5
y_test_d  = np.expm1(y_test)  / 1e5
print(f"alvo em dólares ÷ 1e5: {y_train_d.min():.3f} a {y_train_d.max():.3f}")
print(f"  (equivale a ${y_train_d.min()*1e5:,.0f} a ${y_train_d.max()*1e5:,.0f})\n")

res_div = varredura(y_train_d, y_test_d, "div1e5", "varredura_div1e5.csv")
print(f"\nconcluído: {len(res_div)} configurações")

alvo em dólares ÷ 1e5: 0.349 a 7.550
  (equivale a $34,900 a $755,000)

  [div1e5] therm= 5 concluído  (35 configs, 0.4 min)
  [div1e5] therm= 6 concluído  (84 configs, 1.1 min)
  [div1e5] therm= 7 concluído  (119 configs, 1.7 min)
  [div1e5] therm= 8 concluído  (154 configs, 2.4 min)
  [div1e5] therm= 9 concluído  (210 configs, 3.5 min)
  [div1e5] therm=10 concluído  (259 configs, 4.6 min)
  [div1e5] therm=11 concluído  (294 configs, 5.5 min)
  [div1e5] therm=12 concluído  (357 configs, 7.3 min)
  [div1e5] therm=13 concluído  (392 configs, 8.4 min)
  [div1e5] therm=14 concluído  (441 configs, 10.0 min)
  [div1e5] therm=15 concluído  (518 configs, 12.6 min)
  [div1e5] therm=16 concluído  (560 configs, 14.2 min)
  [div1e5] therm=17 concluído  (588 configs, 15.6 min)
  [div1e5] therm=18 concluído  (658 configs, 18.8 min)
  [div1e5] therm=19 concluído  (686 configs, 20.4 min)
  [div1e5] therm=20 concluído  (742 configs, 23.3 min)
  [div1e5] therm=21 concluído  (812 configs, 27.1 min)
  [d

### Resultado das duas varreduras

In [26]:
res = pd.concat([res_log, res_div], ignore_index=True)
res.to_csv("varredura_completa.csv", index=False)

TESE = {"ReW": 0.278, "CReW": 0.194, "nTRN": 0.302}

for rot in ["log1p", "div1e5"]:
    sub = res[res.escala == rot]
    b = sub.loc[sub.MAE.idxmin()]
    print(f"\n══ escala {rot} ══")
    print(f"  melhor global : MAE {b.MAE:.4f} ± {b.MAE_dp:.4f}   "
          f"therm={b.therm} address={b.address} média={b.media}")
    md = sub[sub.media == "Median"]
    bm = md.loc[md.MAE.idxmin()]
    print(f"  melhor Median : MAE {bm.MAE:.4f} ± {bm.MAE_dp:.4f}   "
          f"therm={bm.therm} address={bm.address}")
    print(f"  tese/artigo (ReW, Median): MAE {TESE['ReW']:.3f}")

print("\n\nMELHOR MAE POR MÉDIA, nas duas escalas\n")
print(res.pivot_table(index="media", columns="escala", values="MAE",
                      aggfunc="min").round(4).to_string())


══ escala log1p ══
  melhor global : MAE 0.1355 ± 0.0018   therm=27 address=54 média=HarmonicMean
  melhor Median : MAE 0.1891 ± 0.0030   therm=25 address=20
  tese/artigo (ReW, Median): MAE 0.278

══ escala div1e5 ══
  melhor global : MAE 0.2524 ± 0.0038   therm=28 address=56 média=GeometricMean
  melhor Median : MAE 0.3540 ± 0.0066   therm=30 address=24
  tese/artigo (ReW, Median): MAE 0.278


MELHOR MAE POR MÉDIA, nas duas escalas

escala            div1e5   log1p
media                           
ExponentialMean   0.2711  0.1383
GeometricMean     0.2524  0.1371
HarmonicMean      0.2567  0.1355
HarmonicPower(2)  0.2668  0.1359
Median            0.3540  0.1891
PowerMean(2)      0.2542  0.1380
SimpleMean        0.2954  0.1654


In [27]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=False)
for a, rot in zip(ax, ["log1p", "div1e5"]):
    sub = res[res.escala == rot]
    for nome, g in sub.groupby("media"):
        best = g.groupby("therm")["MAE"].min()
        a.plot(best.index, best.values, marker="o", ms=3, lw=1.3, label=nome,
               color=COR_KAL if nome == "Median" else None,
               zorder=5 if nome == "Median" else 2,
               alpha=1.0 if nome == "Median" else .55)
    a.axhline(TESE["ReW"], color=COR_REW, ls="--", lw=1.4, label="tese: 0.278")
    a.set(xlabel="thermometer size", ylabel="melhor MAE",
          title=f"escala {rot}")
    a.legend(frameon=False, fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

NameError: name 'plt' is not defined

---
## 7. A configuração vencedora anunciada

O artigo e a tese informam que a melhor ReW no House Prices usou **Median**, mas
não informam o thermometer nem o address. A célula abaixo isola a `Median` e
mostra o MAE dela em toda a grade, nas duas escalas, para localizar onde ela
chega em 0.278 (se chegar).

In [28]:
md = res[res.media == "Median"].copy()

for rot in ["log1p", "div1e5"]:
    sub = md[md.escala == rot]
    print(f"\n══ Median, escala {rot} ══")
    print(f"  faixa de MAE: {sub.MAE.min():.4f} a {sub.MAE.max():.4f}")
    perto = sub.iloc[(sub.MAE - 0.278).abs().argsort()[:5]]
    print(f"  as 5 configurações mais próximas de 0.278:")
    print(perto[["therm", "address", "n_rams", "MAE", "MAE_dp"]].to_string(index=False))

print("\n\nUm detalhe da Tabela 2 do artigo que vale reter: o desvio padrão")
print("reportado é ZERO nas dez repetições. Com o mapping sorteado a cada")
print("instância, dez execuções deveriam variar. Compare com a coluna MAE_dp")
print("acima, que é o desvio das nossas dez repetições.")


══ Median, escala log1p ══
  faixa de MAE: 0.1891 a 0.3346
  as 5 configurações mais próximas de 0.278:
 therm  address  n_rams      MAE   MAE_dp
    13       52      83 0.274792 0.009184
    21       42     166 0.281763 0.006288
    27        6    1494 0.272833 0.001165
    30        6    1660 0.272712 0.000736
    21        6    1162 0.272675 0.001422

══ Median, escala div1e5 ══
  faixa de MAE: 0.3540 a 0.6568
  as 5 configurações mais próximas de 0.278:
 therm  address  n_rams      MAE   MAE_dp
    30       24     415 0.354035 0.006565
    15       20     249 0.355736 0.004993
    25       20     415 0.358539 0.006357
     5       20      83 0.360712 0.004431
    18       24     249 0.361411 0.007931


Um detalhe da Tabela 2 do artigo que vale reter: o desvio padrão
reportado é ZERO nas dez repetições. Com o mapping sorteado a cada
instância, dez execuções deveriam variar. Compare com a coluna MAE_dp
acima, que é o desvio das nossas dez repetições.
